# Load COVID-19 Conjoint Data & Pair Choice Sets

In [49]:
# Step 1: Load and validate the raw Kreps et al. (2020) COVID-19 conjoint dataset
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm

DATA_PATH = Path("../../data/covid/dataverse_files/Kreps_etal_vax_replication_data.dta")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing dataset at {DATA_PATH.resolve()}")

# Read Stata file with numeric codes intact (e.g. efficacy_n in {1,2,3}) rather than converting to pandas Categoricals
raw_df = pd.read_stata(DATA_PATH, convert_categoricals=False)

# Verify full study size: 1,971 respondents x 5 choice tasks x 2 vaccine profiles per task = 19,710 rows
if len(raw_df) != 19710 or raw_df["respondent"].nunique() != 1971:
    raise ValueError(f"Unexpected dataset shape: {raw_df.shape}, respondents={raw_df['respondent'].nunique()}")

raw_df["respondent"] = raw_df["respondent"].astype(int)
raw_df["choice_set"] = raw_df["choice_set"].astype(int)

# choice_set is a 2-digit code {11,12,21,22,31,32,41,42,51,52}: tens digit = task 1..5, ones digit = Vaccine A (1) or B (2)
raw_df["pair_id"] = raw_df["choice_set"] // 10
raw_df["option_ab"] = raw_df["choice_set"] % 10

if set(raw_df["pair_id"].unique()) != {1, 2, 3, 4, 5} or set(raw_df["option_ab"].unique()) != {1, 2}:
    raise ValueError("Unexpected choice_set encoding")

# Sort so every consecutive pair of rows (2*i, 2*i+1) corresponds to (Vaccine A, Vaccine B) of the same choice task
eval_df = raw_df.sort_values(["respondent", "pair_id", "option_ab"]).reset_index(drop=True)
print(
    f"Loaded {len(eval_df):,} profile evaluations across {eval_df['respondent'].nunique():,} respondents "
    f"({len(eval_df) // 2:,} paired choice sets)"
)
eval_df[[
    "respondent", "choice_set", "pair_id", "option_ab", "vaxbin",
    "efficacy", "duration", "majorside", "minorside", "fda", "origin", "endorsed",
]].head(6)

Loaded 19,710 profile evaluations across 1,971 respondents (9,855 paired choice sets)


,respondent,choice_set,pair_id,option_ab,vaxbin,efficacy,duration,majorside,minorside,fda,origin,endorsed
0,1,11,1,1,1.0,90%,5 years,"1 in 10,000",1 in 30,The vaccine has been approved and licensed by ...,United Kingdom,President Donald Trump
1,1,12,1,2,0.0,70%,1 year,"1 in 1,000,000",1 in 10,The vaccine has been approved and licensed by ...,China,World Health Organization
2,1,21,2,1,1.0,90%,5 years,"1 in 10,000",1 in 10,The vaccine has been approved and licensed by ...,United Kingdom,US Centers for Disease Control and Prevention
3,1,22,2,2,0.0,50%,5 years,"1 in 10,000",1 in 10,The vaccine has received an emergency use auth...,United States,World Health Organization
4,1,31,3,1,0.0,50%,5 years,"1 in 1,000,000",1 in 30,The vaccine has received an emergency use auth...,United States,Vice President Joe Biden
5,1,32,3,2,1.0,90%,1 year,"1 in 1,000,000",1 in 30,The vaccine has received an emergency use auth...,China,Vice President Joe Biden


In [50]:
# Step 2: Pair Option A and Option B into choice sets and construct 3-way human choice targets (A, B, Neither)
# Even indices (0, 2, 4, ...) are Vaccine A profiles; odd indices (1, 3, 5, ...) are Vaccine B profiles
df_A = eval_df.iloc[0::2].reset_index(drop=True)
df_B = eval_df.iloc[1::2].reset_index(drop=True)

# Confirm each row k in df_A and df_B belongs to the exact same respondent and choice task
if not (df_A["respondent"].equals(df_B["respondent"]) and df_A["pair_id"].equals(df_B["pair_id"])):
    raise ValueError("Mismatched (respondent, pair_id) between Option A and Option B rows")
if not ((df_A["option_ab"] == 1).all() and (df_B["option_ab"] == 2).all()):
    raise ValueError("Expected option_ab == 1 for df_A and option_ab == 2 for df_B")

# vaxbin is 1 if that profile was chosen, 0 otherwise; Option C (neither vaccine chosen) is 1 when both A and B are 0
y_human_A = df_A["vaxbin"].to_numpy(dtype=np.float64)
y_human_B = df_B["vaxbin"].to_numpy(dtype=np.float64)
y_human_C = 1.0 - y_human_A - y_human_B

# Ensure every choice set is a valid one-hot choice among {A, B, C} (no respondent can choose both A and B)
valid_pairs = set(zip(y_human_A.tolist(), y_human_B.tolist(), y_human_C.tolist()))
if not valid_pairs.issubset({(1.0, 0.0, 0.0), (0.0, 1.0, 0.0), (0.0, 0.0, 1.0)}):
    raise ValueError(f"Invalid 3-way choice combinations: {valid_pairs}")

# Summarize empirical choice frequencies and shares across all 9,855 choice sets
choice_summary = pd.DataFrame({
    "Option": ["A (Vaccine A)", "B (Vaccine B)", "C (Neither)"],
    "Count": [int(y_human_A.sum()), int(y_human_B.sum()), int(y_human_C.sum())],
    "Share": [float(y_human_A.mean()), float(y_human_B.mean()), float(y_human_C.mean())],
})
choice_summary

,Option,Count,Share
0,A (Vaccine A),4069,0.412887
1,B (Vaccine B),3715,0.376966
2,C (Neither),2071,0.210147


# Prompt

standardized for all generative LLMs based on Wang 2026

In [51]:
# Step 3: Subsample 200 respondents (1,000 choice sets) matching Wang (2026) LLM-only sample size (n = 1,000)
N_LLM_RESPONDENTS = 200
SUBSAMPLE_SEED = 42

# Uniformly sample 200 unique respondent IDs without replacement (5 choice sets per respondent = 1,000 pairs)
all_respondents = np.sort(df_A["respondent"].unique())
rng = np.random.default_rng(SUBSAMPLE_SEED)
sampled_respondents = np.sort(rng.choice(all_respondents, size=N_LLM_RESPONDENTS, replace=False))

# Boolean mask selecting all 5 choice sets for each of the 200 sampled respondents
sub_mask = df_A["respondent"].isin(sampled_respondents).to_numpy()
sub_df_A = df_A.loc[sub_mask].reset_index(drop=True)
sub_df_B = df_B.loc[sub_mask].reset_index(drop=True)

if len(sub_df_A) != 1000 or sub_df_A["respondent"].nunique() != 200:
    raise ValueError(f"Expected 1,000 choice sets across 200 respondents, got {len(sub_df_A)} rows")

print(f"Sampled {sub_df_A['respondent'].nunique()} respondents -> {len(sub_df_A):,} choice sets (seed={SUBSAMPLE_SEED})")
print("First 10 sampled respondent IDs:", sampled_respondents[:10].tolist())

Sampled 200 respondents -> 1,000 choice sets (seed=42)
First 10 sampled respondent IDs: [14, 45, 59, 67, 71, 81, 112, 116, 124, 141]


In [52]:
# Step 7: Build Kev-style respondent persona blocks using apply_persona_mapping(..., 'covid') from code/data/persona.py
import importlib
import sys
if "../.." not in sys.path:
    sys.path.insert(0, "../..")

import data.persona as _persona_mod
importlib.reload(_persona_mod)
from data.persona import DATASET_PERSONA_COLS, PERSONA_NAMES, apply_persona_mapping

# Take one row per respondent (choice_set == 11) and map all 9 COVID persona attributes to canonical global levels
resp_raw = eval_df[eval_df["choice_set"] == 11].sort_values("respondent").reset_index(drop=True)
KEV_COVID_PERSONA_COLS = DATASET_PERSONA_COLS["covid"]
resp_persona_df = apply_persona_mapping(resp_raw, "covid")[["respondent", *KEV_COVID_PERSONA_COLS]]

# Build newline-separated 'Label: Value' state string per respondent (omitting None/NaN when gender == 'Prefer not to say')
resp_state_by_id: dict[int, str] = {}
for row in resp_persona_df.to_dict("records"):
    lines = [
        f"{PERSONA_NAMES[col]}: {row[col]}"
        for col in KEV_COVID_PERSONA_COLS
        if row[col] is not None and pd.notna(row[col])
    ]
    resp_state_by_id[int(row["respondent"])] = "\n".join(lines)

resp_persona_df.head(5)

,respondent,c_alphan,SEX,AGE,EDULEVEL,RELIGGRP,PARTY_LR,US_PRTY,US_INC,RACE
0,1,United States,Female,30-49,Associate's degree,Agnostic,Somewhat liberal,Independent,"$60,000 - $79,999",White
1,2,United States,Female,30-49,Less than high school,Nothing in particular,Moderate,Independent,"Below $20,000",White
2,3,United States,Female,30-49,High school or some college,Other,Somewhat liberal,Democrat,"Below $20,000",White
3,4,United States,Male,18-29,Associate's degree,Catholic,Very liberal,Democrat,"$20,000 - $39,999",Black
4,5,United States,Female,65+,High school or some college,Jewish,Liberal,Democrat,"$60,000 - $79,999",White


In [53]:
# Step 8: Build the two prompt sets (1. Random Persona, 2. Actual Respondent Persona) on the 1,000 sampled pairs
# Exact mapping from Stata `fda` strings to Wang (2026) Section 5.2.1 minimalist Authorization descriptions
FDA_WANG_TEXT_MAP = {
    "The vaccine has been approved and licensed by the US Food and Drug Administration.": (
        "Approved and licensed by the US Food and Drug Administration."
    ),
    "The vaccine has received an emergency use authorization from the US Food and Drug Administration.  This allows the expedited use of promising drugs that the FDA has found it reasonable to believe may be effective in combatting the virus": (
        "Received an emergency use authorization from the US Food and Drug Administration.  "
        "This allows the expedited use of promising drugs that the FDA has found it reasonable "
        "to believe may be effective in combatting the virus."
    ),
}


def format_wang_option(letter: str, row: pd.Series | dict) -> str:
    """Format a single vaccine option (A or B) using Wang (2026) Section 5.2.1 minimalist feature wording."""
    fda_raw = row["fda"]
    if fda_raw not in FDA_WANG_TEXT_MAP:
        raise ValueError(f"Unexpected fda text: {fda_raw!r}")
    fda_text = FDA_WANG_TEXT_MAP[fda_raw]
    return (
        f"{letter}: Efficacy: {row['efficacy']}. "
        f"Protective duration: {row['duration']}. "
        f"Major side effect: {row['majorside']}. "
        f"Minor side effect: {row['minorside']}. "
        f"Authorization: {fda_text} "
        f"Origin: {row['origin']}. "
        f"Endorsement: {row['endorsed']}."
    )


def build_options_block(row_a: pd.Series | dict, row_b: pd.Series | dict) -> str:
    """Build Wang (2026) Input (II): text description of Options A, B, and C."""
    opt_a = format_wang_option("A", row_a)
    opt_b = format_wang_option("B", row_b)
    return (
        "There are three options for you:\n"
        f"{opt_a}\n"
        f"{opt_b}\n"
        "C: You choose neither A nor B."
    )


# Set 1 (Random Persona): exact Wang (2026) Section 5.2.1 system & user prompts
RANDOM_SYSTEM_PROMPT = (
    "You should act like a random person deciding whether or not to receive a COVID-19 vaccination."
)


def build_random_persona_prompt(row_a: pd.Series | dict, row_b: pd.Series | dict) -> dict[str, object]:
    """Construct (system_prompt, user_prompt) for the Wang (2026) random-person baseline."""
    options_block = build_options_block(row_a, row_b)
    user_prompt = (
        f"{options_block}\n\n"
        "Which option would you choose as a random person? Your response should be a single letter A, B, or C."
    )
    return {
        "respondent": int(row_a["respondent"]),
        "pair_id": int(row_a["pair_id"]),
        "system_prompt": RANDOM_SYSTEM_PROMPT,
        "user_prompt": user_prompt,
    }


# Set 2 (Actual Respondent Persona): Wang (2026) framework + Kev-style 'Key: Value' persona block
def build_respondent_persona_prompt(row_a: pd.Series | dict, row_b: pd.Series | dict) -> dict[str, object]:
    """Construct (system_prompt, user_prompt) combining Wang (2026) task instructions with Kev persona state."""
    resp_id = int(row_a["respondent"])
    persona_state = resp_state_by_id[resp_id]
    options_block = build_options_block(row_a, row_b)
    system_prompt = (
        "You should act like a person with the following characteristics deciding whether or not "
        "to receive a COVID-19 vaccination:\n"
        f"{persona_state}"
    )
    user_prompt = (
        f"{options_block}\n\n"
        "Which option would you choose as this person? Your response should be a single letter A, B, or C."
    )
    return {
        "respondent": resp_id,
        "pair_id": int(row_a["pair_id"]),
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
    }


sub_records_A = sub_df_A.to_dict("records")
sub_records_B = sub_df_B.to_dict("records")

random_persona_prompts = [build_random_persona_prompt(ra, rb) for ra, rb in zip(sub_records_A, sub_records_B)]
respondent_persona_prompts = [build_respondent_persona_prompt(ra, rb) for ra, rb in zip(sub_records_A, sub_records_B)]

print(f"Built {len(random_persona_prompts):,} random-persona prompts and {len(respondent_persona_prompts):,} respondent-persona prompts.\n")
print("=== 1. RANDOM PERSONA SAMPLE (Sampled Pair 0) ===")
print("[System Prompt]\n" + random_persona_prompts[0]["system_prompt"])
print("\n[User Prompt]\n" + random_persona_prompts[0]["user_prompt"])
print("\n" + "=" * 70 + "\n")
print("=== 2. ACTUAL RESPONDENT PERSONA SAMPLE (Sampled Pair 0) ===")
print("[System Prompt]\n" + respondent_persona_prompts[0]["system_prompt"])
print("\n[User Prompt]\n" + respondent_persona_prompts[0]["user_prompt"])

Built 1,000 random-persona prompts and 1,000 respondent-persona prompts.

=== 1. RANDOM PERSONA SAMPLE (Sampled Pair 0) ===
[System Prompt]
You should act like a random person deciding whether or not to receive a COVID-19 vaccination.

[User Prompt]
There are three options for you:
A: Efficacy: 50%. Protective duration: 5 years. Major side effect: 1 in 10,000. Minor side effect: 1 in 10. Authorization: Approved and licensed by the US Food and Drug Administration. Origin: United Kingdom. Endorsement: World Health Organization.
B: Efficacy: 50%. Protective duration: 5 years. Major side effect: 1 in 10,000. Minor side effect: 1 in 30. Authorization: Received an emergency use authorization from the US Food and Drug Administration.  This allows the expedited use of promising drugs that the FDA has found it reasonable to believe may be effective in combatting the virus. Origin: United Kingdom. Endorsement: Vice President Joe Biden.
C: You choose neither A nor B.

Which option would you choos

In [54]:
len(respondent_persona_prompts), len(random_persona_prompts)

(1000, 1000)

# Query Gemini & Save JSONL Datasets

In [55]:
# Step 9: Import Gemini dataset collection helpers and configure persistent data directory
import importlib
import gemini_api

importlib.reload(gemini_api)
from gemini_api import parse_letter_choices, query_gemini_prompts

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [65]:
# Step 10: Query Gemini on Set 1 (Random Persona) and progressively save dataset to disk
# Set model, thinking_level ("MINIMAL", "LOW", "MEDIUM", "HIGH", or None), and n_personas (e.g. 100 or 200)
model = "gemini-4-argon"
thinking_level = "HIGH"
# N_QUERY_PERSONAS = 10
N_QUERY_PERSONAS = None

raw_random_responses = await query_gemini_prompts(
    random_persona_prompts,
    output_path=DATA_DIR / f"{model}_{thinking_level}_random.jsonl",
    model=model,
    thinking_level=thinking_level,
    n_personas=N_QUERY_PERSONAS,
)
pd.Series([r.strip() for r in raw_random_responses], name="choice").value_counts(dropna=False).to_frame("count")

[gemini-4-argon_HIGH_random.jsonl] model=gemini-4-argon, thinking_level=HIGH | Target: 1000 prompts (10 already collected, 990 remaining)


gemini-4-argon_HIGH_random:   0%|          | 0/990 [00:00<?, ?req/s]

,count
choice,
A,533
B,451
"Deciding between these options depends heavily on an individual's personal priorities regarding risk, efficacy, regulatory standards, and trust in public figures, meaning different people would arrive at different choices:\n\n* **Reasons to choose Option A:** A person prioritizing strong protection against the virus and rigorous regulatory standards would likely favor Option A due to its 90% efficacy and full FDA approval. Its domestic origin and endorsement by President Donald Trump might also increase trust for certain individuals, provided they are willing to accept a higher risk of major side effects (1 in 10,000).\n* **Reasons to choose Option B:** A person whose primary concern is minimizing the risk of severe adverse reactions would likely prefer Option B, as the chance of a major side effect is significantly lower (1 in 1,000,000). This option would also appeal to those who place greater trust in an endorsement from Vice President Joe Biden, even though the vaccine offers lower efficacy (50%) and relies on an Emergency Use Authorization.\n* **Reasons to choose Option C:** A person might opt against both vaccines if they find the side-effect risk of Option A too high and the efficacy or expedited authorization of Option B insufficient. Others might choose this option due to general vaccine hesitancy or skepticism toward political endorsements of medical products. \n\nBecause a person's choice depends on how they weigh these competing factors, there is no single option that represents what a random individual would decide.",1
"Choosing between these options depends entirely on an individual's personal priorities, risk tolerance, and trust in different institutions, so there is no single choice that a random person would make:\n\n* **Someone who chooses A** would likely prioritize long-term protection (5 years) and the lowest possible risk of a major side effect (1 in 1,000,000), while trusting the endorsement of the World Health Organization, even if it means accepting lower efficacy (50%) and an Emergency Use Authorization.\n* **Someone who chooses B** would likely prioritize high efficacy (90%), full FDA approval, and a lower chance of minor side effects (1 in 30), or value the endorsement of Joe Biden, while being willing to accept a shorter duration of protection (1 year) and a higher risk of major side effects (1 in 10,000).\n* **Someone who chooses C** might find the trade-offs in both vaccines unacceptable—such as the low efficacy of Option A or the higher major side-effect risk of Option B—or they may have reservations about vaccines developed in China, or prefer not to receive a COVID-19 vaccine at all.",1
"Choosing between these options depends heavily on an individual's personal priorities, risk tolerance, and trust in various institutions, meaning different people would arrive at different decisions:\n\n* **Reasons to choose Option A:** A person prioritizing medical outcomes would likely favor this option because it offers the highest protection (90% efficacy) and the lowest risk of a major side effect (1 in 1,000,000). For supporters of Donald Trump, his endorsement would also be a positive factor. However, others might be deterred by the fact that it only has an Emergency Use Authorization rather than full approval, or they may have concerns about a vaccine developed in China.\n\n* **Reasons to choose Option B:** A person who places high value on institutional vetting and domestic production might prefer this option, as it has full FDA approval, originated in the United States, and is endorsed by the CDC. On the other hand, many people would be hesitant to choose a vaccine that is only 50% effective and carries a notably higher risk of major side effects (1 in 10,000) compared to Option A.\n\n* **Reasons to choose Option C:** A person might opt out of both vaccines if they find the trade-offs unacceptable—for instance, if they do not trust a vaccine from China with only emergency

In [72]:
df_random = pd.read_json(DATA_DIR / f"{model}_{thinking_level}_random.jsonl", lines=True)
print(df_random.loc[997]['reasoning_text'])
print('+'*50)
print(df_random.loc[997]['raw_response'])

**My Thought Process on Choosing a Hypothetical Vaccine**

Okay, so the user wants me to step into the shoes of a "random person" and make a very specific choice: either vaccine A, vaccine B, or neither (C). The catch is that I have to respond with just a single letter, A, B, or C. This feels like a survey question, and honestly, it's designed to be tricky, especially given the details.

Here's the breakdown of what I'm looking at:

*   **Vaccine A:** Endorsed by the US Centers for Disease Control and Prevention (CDC).
*   **Vaccine B:** Endorsed by President Donald Trump.
*   **All Other Attributes (Identical for A & B):**
    *   Efficacy: 50%
    *   Protective Duration: 5 years
    *   Major Side Effect Rate: 1 in 1,000,000 (very rare)
    *   Minor Side Effect Rate: 1 in 30 (noticeable, but not severe)
    *   Authorization: FDA Emergency Use Authorization (EUA)
    *   Origin: UK

Now, this setup is a classic example of how researchers in political science and public health study

In [66]:
# Step 12: Query Gemini on Set 2 (Actual Respondent Persona) and progressively save dataset to disk
raw_persona_responses = await query_gemini_prompts(
    respondent_persona_prompts,
    output_path=DATA_DIR / f"{model}_{thinking_level}_persona.jsonl",
    model=model,
    thinking_level=thinking_level,
    n_personas=N_QUERY_PERSONAS,
)
pd.Series([r.strip() for r in raw_persona_responses], name="choice").value_counts(dropna=False).to_frame("count")

[gemini-4-argon_HIGH_persona.jsonl] model=gemini-4-argon, thinking_level=HIGH | Target: 1000 prompts (0 already collected, 1000 remaining)


gemini-4-argon_HIGH_persona:   0%|          | 0/1000 [00:00<?, ?req/s]

,count
choice,
A,481
B,428
C,91
